In [1]:
%reset -f

# Endogeneity and Instrumental Variables





## load all packages

In [25]:
import numpy             as np
import statsmodels.api   as sm
import pandas            as pd
import seaborn           as sns

from linearmodels.iv            import IV2SLS
from linearmodels               import PooledOLS
from statsmodels.iolib.summary2 import summary_col
from collections                import OrderedDict
from linearmodels.iv.results    import compare

## load data

In [10]:
df = pd.read_stata("/home/michael/Projects/regression_analysis/2_endogenity_iv/maketable4/maketable4.dta")
## only keep base sample - only reproducing some esitmates
df = df[df['baseco'] == 1]

df['oth_cont'] = np.where((df['shortnam'] == "AUS") | 
                          (df['shortnam'] == "MLT") | 
                          (df['shortnam'] == "NZL"), 1,0)

Below is a table describing the data. 
| Name | Label |
| :--- | :--- |
| `shortnam` | 3-letter country name |
| `africa` | Africa |
| `latitude` | Absolute value of latitude |
| `rich4` | ... TBC |
| `avexpr` | Exp Risk |
| `logpgp95` |  Log GDP per capita (PPP) in 1995  |
| `logem4` |  Original Log Settler Mortality  |
| `asia` | Asia |
| `loghjypl` | ... TBC |
| `baseco` | =1 if part of base sample |
| `oth_cont`| =1 if other continent | 

Note that this data was downloaded from a MIT dropbox on 16/6/2026 -https://www.dropbox.com/scl/fi/3yuv9j514zuajzjfluoc1/maketable4.zip?rlkey=pq9l7bxktw1iqxe6fmoh26g79&e=1&dl=0

In [11]:
df.describe()

,africa,lat_abst,rich4,avexpr,logpgp95,logem4,asia,loghjypl,baseco,oth_cont
count,64.000000,64.000000,64.000000,64.000000,64.000000,64.000000,64.000000,61.000000,64.0,64.000000
mean,0.421875,0.181103,0.062500,6.515625,8.062237,4.657031,0.140625,-1.934052,1.0,0.046875
std,0.497763,0.132667,0.243975,1.468647,1.043359,1.257984,0.350382,0.980744,0.0,0.213042
min,0.000000,0.000000,0.000000,3.500000,6.109248,2.145931,0.000000,-3.540459,1.0,0.000000
25%,0.000000,0.088889,0.000000,5.613636,7.299728,4.232656,0.000000,-2.748872,1.0,0.000000
50%,0.000000,0.152778,0.000000,6.477273,7.949796,4.358630,0.000000,-1.864330,1.0,0.000000
75%,1.000000,0.258333,0.000000,7.352273,8.848779,5.519177,0.000000,-1.331806,1.0,0.000000
max,1.000000,0.666667,1.000000,10.000000,10.215740,7.986165,1.000000,0.000000,1.0,1.000000


In [33]:
# table 2 column 2
m_2_2 = sm.OLS(df['logpgp95'], sm.add_constant(df['avexpr'])).fit()
print(m_2_2.summary())

                            OLS Regression Results                            
Dep. Variable:               logpgp95   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.533
Method:                 Least Squares   F-statistic:                     72.82
Date:                Tue, 16 Jun 2026   Prob (F-statistic):           4.72e-12
Time:                        11:59:12   Log-Likelihood:                -68.168
No. Observations:                  64   AIC:                             140.3
Df Residuals:                      62   BIC:                             144.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6604      0.409     11.408      0.0

In [34]:
# table 4 Panel B column 1 
m_4_B_1 = sm.OLS(df['avexpr'], sm.add_constant(df['logem4'])).fit()
print(m_4_B_1.summary())

                            OLS Regression Results                            
Dep. Variable:                 avexpr   R-squared:                       0.270
Model:                            OLS   Adj. R-squared:                  0.258
Method:                 Least Squares   F-statistic:                     22.95
Date:                Tue, 16 Jun 2026   Prob (F-statistic):           1.08e-05
Time:                        11:59:13   Log-Likelihood:                -104.83
No. Observations:                  64   AIC:                             213.7
Df Residuals:                      62   BIC:                             218.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.3414      0.611     15.296      0.0

In [17]:
# table 4 Panel A column 1 
print(IV2SLS(dependent=df['logpgp95'],
              exog=df['baseco'],
            endog=df[['avexpr']],
            instruments=df[['logem4']]).fit())

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.1870
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1739
No. Observations:                  64   F-statistic:                    28.754
Date:                Tue, Jun 16 2026   P-value (F-stat)                0.0000
Time:                        11:34:45   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
baseco         1.9097     1.1740     1.6267     0.1038     -0.3912      4.2106
avexpr         0.9443     0.1761     5.3623     0.00

In [19]:
# table 4 Panel A column 7
print(IV2SLS(dependent=df['logpgp95'],
              exog=df[['baseco', 'asia', 'africa', 'oth_cont']],
            endog=df[['avexpr']],
            instruments=df[['logem4']]).fit())

                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.2286
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1763
No. Observations:                  64   F-statistic:                    36.376
Date:                Tue, Jun 16 2026   P-value (F-stat)                0.0000
Time:                        11:37:31   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
baseco         2.0324     2.1905     0.9278     0.3535     -2.2609      6.3257
asia          -0.9242     0.3705    -2.4947     0.01

In [16]:
# table 4 Panel B column 8
print(IV2SLS(dependent=df['logpgp95'],
              exog=df[['baseco', 'lat_abst', 'asia', 'africa', 'oth_cont']],
            endog=df[['avexpr']],
            instruments=df[['logem4']]).fit())


                          IV-2SLS Estimation Summary                          
Dep. Variable:               logpgp95   R-squared:                      0.0108
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0745
No. Observations:                  64   F-statistic:                    28.443
Date:                Tue, Jun 16 2026   P-value (F-stat)                0.0000
Time:                        11:32:35   Distribution:                  chi2(5)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
baseco         1.4405     3.0736     0.4687     0.6393     -4.5837      7.4646
lat_abst      -1.1782     1.7917    -0.6576     0.51

In [38]:
# ignore this really uglly code - it makes a pretty table tho
results = OrderedDict()
results['T.2 col 2'] = IV2SLS(dependent=df['logpgp95'], exog=df[['baseco', "avexpr"]], endog=None, instruments= None).fit()
#above is the same as sm.OLS(df['logpgp95'], sm.add_constant(df['avexpr'])).fit()
results['T. 4 Panel B col 1'] = IV2SLS(dependent=df['avexpr'], exog=df[['baseco','logem4']], endog=None, instruments= None).fit()
# above is the same as sm.OLS(df['avexpr'], sm.add_constant(df['logem4'])).fit()
results['T. 4 Panel A col 1'] = IV2SLS(dependent=df['logpgp95'], exog=df['baseco'], endog=df[['avexpr']], instruments=df[['logem4']]).fit()
results['T. 4 Panel A col 7'] = IV2SLS(dependent=df['logpgp95'], exog=df[['baseco', 'asia', 'africa', 'oth_cont']], endog=df[['avexpr']], instruments=df[['logem4']]).fit()
results['T. 4 Panel B col 8'] = IV2SLS(dependent=df['logpgp95'],exog=df[['baseco', 'lat_abst', 'asia', 'africa', 'oth_cont']], endog=df[['avexpr']], instruments=df[['logem4']]).fit()
print(compare(results,precision='std_errors',stars=True))

                                               Model Comparison                                               
                         T.2 col 2 T. 4 Panel B col 1 T. 4 Panel A col 1 T. 4 Panel A col 7 T. 4 Panel B col 8
--------------------------------------------------------------------------------------------------------------
Dep. Variable             logpgp95             avexpr           logpgp95           logpgp95           logpgp95
Estimator                      OLS                OLS            IV-2SLS            IV-2SLS            IV-2SLS
No. Observations                64                 64                 64                 64                 64
Cov. Est.                   robust             robust             robust             robust             robust
R-squared                   0.5401             0.2701             0.1870             0.2286             0.0108
Adj. R-squared              0.5327             0.2584             0.1739             0.1763            -0.0745
F

# Instrument Tests
Lets test for the IV that has the best adjusted $R^2$, Table 4 Panel A Column 7.


In [39]:
testing_mod = IV2SLS(dependent=df['logpgp95'], exog=df[['baseco', 'asia', 'africa', 'oth_cont']], endog=df[['avexpr']], instruments=df[['logem4']]).fit()

In [40]:
testing_mod.wu_hausman()

Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 11.8774
P-value: 0.0011
Distributed: F(1,58)
WaldTestStatistic, id: 0x7597fb1527e0

In [41]:
testing_mod.durbin()

Durbin test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 10.8784
P-value: 0.0010
Distributed: chi2(1)
WaldTestStatistic, id: 0x7597f9c69430

In [42]:
testing_mod.wooldridge_regression

Wooldridge's regression test of exogeneity
H0: Endogenous variables are exogenous
Statistic: 6.5582
P-value: 0.0104
Distributed: chi2(1)
WaldTestStatistic, id: 0x7597f9c4e180

In [43]:
testing_mod.wooldridge_score

Wooldridge's score test of exogeneity
H0: Endogenous variables are exogenous
Statistic: 5.4215
P-value: 0.0199
Distributed: chi2(1)
WaldTestStatistic, id: 0x7597f9c489b0